# Oxford-IIIT Pet Pixel-wise Segmentation

This notebook contains the pipeline for pixel-wise segmentation using a graph representation of the full image (i.e., one pixel corresponds to one node).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from torchvision.datasets import OxfordIIITPet

from tqdm.auto import tqdm

from PIL import Image

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

import models

In [ ]:
category = 15
size = 128
threshold = 0.5

In [ ]:
def plot_mask(image, mask):
    mask = np.array(mask)
    masked = np.ma.masked_where(mask == 0, mask)
    plt.imshow(np.array(image))
    plt.imshow(masked, cmap='jet', alpha=0.5)
    plt.axis('off')
    plt.show()
    
def IoU(mask1, mask2):
    mask1 = np.array(mask1, dtype=float)
    mask2 = np.array(mask2, dtype=float)
    intersection = np.logical_and(mask1, mask2)
    union = np.logical_or(mask1, mask2)
    
    return np.sum(intersection) / np.sum(union)

In [ ]:
dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types=("segmentation", "category"),
    download=True
)

test_dataset = OxfordIIITPet(
    root="data",
    split="test",
    target_types=("segmentation", "category"),
    download=True
)

In [ ]:
def preprocess(mask):
    return mask.point(lambda p: 1 if p == 1 or p == 3 else 0)

def downscale(image, mask, size):
    small_image = image.copy()
    small_image.thumbnail((size,size))
    
    small_mask = mask.copy()
    small_mask.thumbnail((size,size))
    
    image = np.array(small_image)
    mask = np.array(small_mask)
    
    return image, mask

def upscale(image, shape):
    pil_img = Image.fromarray(image.astype(np.uint8))
    up_img = pil_img.resize((shape[0], shape[1]), resample=Image.BILINEAR)
    up_img = np.array(up_img)
    
    return up_img

In [ ]:
idx_train = []
for i, (image, (mask, cat)) in tqdm(enumerate(dataset), total=len(dataset)):
    if cat == category:
        idx_train.append(i)

idx_test = []
for i, (image, (mask, cat)) in tqdm(enumerate(test_dataset), total=len(test_dataset)):
    if cat == category:
        idx_test.append(i)

## Data exploration

You can go through the different image in the dataset by changing the variable `i`

In [ ]:
i = 5

image, (mask, cat) = dataset[idx_train[i]]
mask = preprocess(mask)

plot_mask(image, mask)

## Features computation

We compute the features / labels pairs of the train dataset. The features are the RGB pixel representation.

In [ ]:
features = []
labels = []

for i in tqdm(idx_train):
    image, (mask, cat) = dataset[i]
    mask = preprocess(mask)
    image, mask = downscale(image, mask, size)
    mask = mask > threshold
    fg = image[mask]
    features.append(fg)
    labels.append(np.ones(len(fg)))
    bg = image[~mask]
    features.append(bg)
    labels.append(np.zeros(len(bg)))

features = np.concatenate(features)
labels = np.concatenate(labels)

idx_0 = np.where(labels == 0)[0]
idx_1 = np.where(labels == 1)[0]

n = min(len(idx_0), len(idx_1))

idx_0_balanced = np.random.choice(idx_0, size=n, replace=False)
idx_1_balanced = np.random.choice(idx_1, size=n, replace=False)

balanced_idx = np.concatenate([idx_0_balanced, idx_1_balanced])
np.random.shuffle(balanced_idx)

features = features[balanced_idx]
labels = labels[balanced_idx]


## Fit unary model

Here we fit the model that will be the unary. You can choose from 2 model:
- logistic regression
- a model that fits gaussian mixture to the data and output probabilities based on the pdf during inference

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.5, random_state=42, stratify=labels
)

logisticMask = models.LogisticMask(solver="newton-cholesky")
densityMask = models.DensityMask()

logisticMask.fit(X_train, y_train)
densityMask.fit(X_train, y_train)

In [ ]:
model = densityMask # choose model here

y_pred = model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["background", "dog"])
disp.plot(cmap="Blues")
plt.show()


## Prediction with unary only

Most of the work is done by the unary model, here no belief propagation is done, we just take the mask given by the unary model.

In [ ]:
i = 5

image, (mask, cat) = test_dataset[idx_test[i]]
mask = preprocess(mask)
downscaled_image, downscaled_mask = downscale(image, mask, size)

predictions = model.predict_image(downscaled_image)


plot_mask(image, upscale(predictions, image.size))

## Belief propagation

Here we apply belief propagation

In [ ]:
i = 2
max_iter = 50

image, (mask, cat) = test_dataset[idx_test[i]]
mask = preprocess(mask)
downscaled_image, downscaled_mask = downscale(image, mask, size)

predictions = model(downscaled_image, max_iter)
predictions = upscale(predictions, image.size) > threshold

plot_mask(image, predictions)
print(f'iou : {IoU(mask, predictions)}')

### Computing mean IOU on the validation set

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
import numpy as np

def parallel_segmentation(idx, model, max_iter, n_jobs=8):
    results = Parallel(n_jobs=n_jobs)(
        delayed(run_single_image)(k, model, max_iter)
        for k in tqdm(idx)
    )
    return np.array(results)

def run_single_image(k, model, max_iter):
    image, (mask, cat) = test_dataset[k]
    mask = preprocess(mask)
    downscaled_image, downscaled_mask = downscale(image, mask, size)

    predictions = model(downscaled_image, max_iter)
    predictions = upscale(predictions, image.size) > threshold
    return IoU(mask, predictions)

def sequential_segmentation(idx, model, max_iter):
    results = []
    for k in tqdm(idx):
        iou = run_single_image(k, model, max_iter)
        results.append(iou)
    return np.array(results)

In [ ]:
ious = parallel_segmentation(idx_test, model, max_iter)
print("Mean IoU:", ious.mean())